# [Super AI Engineer Season 6] Hackathon Week 4
## 5 Domains Hackathon: Heart Disease Prediction

**Super AI Engineer Season 6 - Level 1 Hackathon**  
- Dataset: Heart Disease Prediction
- จัดทำโดย: 600425-วิศิษฐ์

---
### Notebook Outline
1. Setup & Imports  
2. Data Loading & Initial Inspection  
3. Data Preprocessing
4. Feature Engineering
5. Model Training & Evaluation (AutoGluon)
6. Prediction & Submission Generation


# 1. Setup & Imports
### 1.1 ติดตั้งและนำเข้าไลบรารีที่จำเป็น

In [1]:
!pip -q install autogluon

import numpy as np
import pandas as pd
from autogluon.tabular import TabularDataset, TabularPredictor

import os

KAGGLE_MODE = True

data_dir = "/kaggle/input/competitions/super-ai-engineer-ss-6-heart-disease-prediction"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 8.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 452.1/452.1 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.8/244.8 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 4.5 MB/s eta 0:00

# 2. Data Loading & Initial Inspection
### 2.1 โหลดข้อมูล Train, Test และ Sample Submission

In [2]:
train_df = pd.read_csv(f"{data_dir}/train.csv")
test_df = pd.read_csv(f"{data_dir}/test.csv")
sample_sub = pd.read_csv(f"{data_dir}/sample_submission.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
train_df.head()

Train shape: (223084, 20)
Test shape: (74361, 19)


,ID,History of HeartDisease or Attack,High Blood Pressure,Told High Cholesterol,Cholesterol Checked,Body Mass Index,Smoked 100+ Cigarettes,Diagnosed Stroke,Diagnosed Diabetes,Leisure Physical Activity,Heavy Alcohol Consumption,Health Care Coverage,Doctor Visit Cost Barrier,General Health,Difficulty Walking,Sex,Education Level,Income Level,Age,Vegetable or Fruit Intake (1+ per Day)
0,train_000001,No,Yes,Yes,Yes,40.68,Yes,No,No,No,No,Yes,No,Very Poor,Yes,Female,High school graduate,"$15,000 to less than $20,000",64,Yes
1,train_000002,No,No,No,No,24.36,Yes,No,No,Yes,No,No,Yes,Fair,No,Female,College graduate,"Less than $10,000",50,No
2,train_000003,No,Yes,Yes,Yes,27.33,No,No,No,No,No,Yes,Yes,Very Poor,Yes,Female,High school graduate,"$75,000 or more",61,Yes
3,train_000004,No,Yes,No,Yes,27.01,No,No,No,Yes,No,Yes,No,Good,No,Female,Some high school,"$35,000 to less than $50,000",74,Yes
4,train_000005,NaN,Yes,Yes,Yes,34.56,Yes,No,No,Yes,No,Yes,Yes,Very Poor,Yes,Male,Some high school,"$15,000 to less than $20,000",98,Yes


# 3. Data Preprocessing
### 3.1 ทำความสะอาดข้อมูลเบื้องต้น
สำหรับตารางทำนายโรคหัวใจ เราจะเคลียร์ค่า Null ใน Target column ออกก่อนเป็นอันดับแรก

In [3]:
print("Before dropping null targets:", len(train_df))
train_df = train_df.dropna(subset=['History of HeartDisease or Attack'])
print("After dropping null targets:", len(train_df))

Before dropping null targets: 223084
After dropping null targets: 221390


In [4]:
display(train_df['Age'].describe())
display(train_df['History of HeartDisease or Attack'].value_counts())

count    221390.000000
mean         54.660215
std          17.773171
min          18.000000
25%          42.000000
50%          56.000000
75%          67.000000
max         100.000000
Name: Age, dtype: float64

History of HeartDisease or Attack
No     203322
Yes     18068
Name: count, dtype: int64

# 4. Feature Engineering
### 4.1 การสร้าง Feature ใหม่ (Derived Features)
ใช้เทคนิคเดียวกันกับ AutoGluon Pipeline โดยแปลง Yes/No ให้เป็น Binary (1/0) และสร้างฟีเจอร์สำหรับความเสี่ยงโรคอ้วน (Obesity Risk) คู่กับการจัดหมวดหมู่ BMI

In [5]:
def create_new_features(df):
    df = df.copy()
    
    # --- Helper: Convert Yes/No to binary ---
    binary_map = {"Yes": 1, "No": 0}
    binary_cols = [
        "High Blood Pressure", "Told High Cholesterol", "Cholesterol Checked",
        "Smoked 100+ Cigarettes", "Diagnosed Stroke", "Diagnosed Diabetes",
        "Leisure Physical Activity", "Heavy Alcohol Consumption",
        "Health Care Coverage", "Doctor Visit Cost Barrier",
        "Difficulty Walking", "Vegetable or Fruit Intake (1+ per Day)"
    ]
    
    for col in binary_cols:
        if df[col].dtype == object:
            df[col + "_bin"] = df[col].map(binary_map)
        else:
            df[col + "_bin"] = df[col]
    
    # -------- Feature: BMI Category --------
    def categorize_bmi(bmi): 
        if bmi < 18.5:
            return "Underweight"
        elif bmi < 25:
            return "Normal weight"
        elif bmi < 30:
            return "Overweight"
        else:
            return "Obese"
            
    df['BMI Category'] = df['Body Mass Index'].apply(categorize_bmi)
    
    # -------- Derived Feature: Obesity Risk --------
    df['Obesity Risk'] = ((df['Body Mass Index'] >= 30) & (df['Leisure Physical Activity'] == 'No')).astype(int)
    
    # Drop original columns that have been binarized or deemed redundant
    columns_to_drop = [
        "High Blood Pressure_bin", "Told High Cholesterol_bin", "Cholesterol Checked_bin",
        "Smoked 100+ Cigarettes_bin", "Diagnosed Stroke_bin", "Diagnosed Diabetes_bin",
        "Leisure Physical Activity_bin", "Heavy Alcohol Consumption_bin",
        "Health Care Coverage_bin", "Doctor Visit Cost Barrier_bin",
        "Difficulty Walking_bin", "Vegetable or Fruit Intake (1+ per Day)_bin",
        "Leisure Physical Activity",
    ]
    df = df.drop(columns=columns_to_drop)
    return df

In [6]:
train_df = create_new_features(train_df)
test_df = create_new_features(test_df)
train_df.head(3)

,ID,History of HeartDisease or Attack,High Blood Pressure,Told High Cholesterol,Cholesterol Checked,Body Mass Index,Smoked 100+ Cigarettes,Diagnosed Stroke,Diagnosed Diabetes,Heavy Alcohol Consumption,...,Doctor Visit Cost Barrier,General Health,Difficulty Walking,Sex,Education Level,Income Level,Age,Vegetable or Fruit Intake (1+ per Day),BMI Category,Obesity Risk
0,train_000001,No,Yes,Yes,Yes,40.68,Yes,No,No,No,...,No,Very Poor,Yes,Female,High school graduate,"$15,000 to less than $20,000",64,Yes,Obese,1
1,train_000002,No,No,No,No,24.36,Yes,No,No,No,...,Yes,Fair,No,Female,College graduate,"Less than $10,000",50,No,Normal weight,0
2,train_000003,No,Yes,Yes,Yes,27.33,No,No,No,No,...,Yes,Very Poor,Yes,Female,High school graduate,"$75,000 or more",61,Yes,Overweight,0


### 4.2 ปรับสมดุลข้อมูล (Under-sampling)
ข้อมูลผู้ป่วยโรคหัวใจมีความไม่สมดุลสูง เราจึงสุ่มตัวอย่าง Yes จำนวน 18,068 แถว และ No จำนวน 20,000 แถว เพื่อปรับ Balance ให้ใกล้เคียงกัน

In [7]:
df_yes = train_df[train_df['History of HeartDisease or Attack'] == 'Yes'].sample(n=18068, random_state=42)
df_no = train_df[train_df['History of HeartDisease or Attack'] == 'No'].sample(n=20000, random_state=42)
train_df = pd.concat([df_yes, df_no], ignore_index=True)

print("Class Distribution after Under-sampling:")
print(train_df['History of HeartDisease or Attack'].value_counts())

Class Distribution after Under-sampling:
History of HeartDisease or Attack
No     20000
Yes    18068
Name: count, dtype: int64


# 5. Model Training & Evaluation (AutoGluon)
### 5.1 ตั้งค่า Hyperparameters และเทรนโมเดลด้วย AutoGluon TabularPredictor
เลือกรันตัวแบบ GBM, CAT, XGB และ RF โดยเลือก Preset เป็น `best_quality`

In [8]:
save_path = 'best_model'
hyperparameters = {
    'GBM': [
        {'ag_args_fit': {'num_gpus': 0}},
        {'ag_args_fit': {'num_gpus': 1}}
    ],
    'CAT': [
        {'ag_args_fit': {'num_gpus': 0}},
        {'ag_args_fit': {'num_gpus': 1}}
    ],
    'XGB': [
        {'ag_args_fit': {'num_gpus': 0}},
        {'ag_args_fit': {'num_gpus': 1}}
    ],
    'RF': [
        {'ag_args_fit': {'num_gpus': 0}},
        {'ag_args_fit': {'num_gpus': 1}}
    ],
}
time_limit = 600

In [9]:
predictor = TabularPredictor(label='History of HeartDisease or Attack',
                            problem_type='binary',
                            path=save_path,
                            ).fit(
                                train_df,
                                presets='best_quality',
                                hyperparameters=hyperparameters,
                                time_limit=time_limit
                            )
print("Model Training Complete!")

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Sat Jan 17 11:20:45 UTC 2026
CPU Count:          4
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       29.67 GB / 31.35 GB (94.6%)
Disk Space Avail:   19.50 GB / 19.52 GB (99.9%)
Presets specified: ['best_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` value. Copies of AutoGluon will be fit on subsets of the d

Model Training Complete!


# 6. Prediction & Submission Generation
### 6.1 ประเมินผลระดับความเป็นไปได้ (predict_proba) และปรับ Threshold

In [10]:
# Predict using probabilities
y_pred = predictor.predict_proba(test_df)

# Use custom threshold at 0.505
y_pred['outcome'] = y_pred.apply(lambda row: 'Yes' if row['Yes'] > 0.505 else 'No', axis=1)
print("Prediction Label Distribution:\n", y_pred['outcome'].value_counts())

Prediction Label Distribution:
 outcome
No     50155
Yes    24206
Name: count, dtype: int64


### 6.2 บันทึกไฟล์สำหรับส่งบน Kaggle

In [11]:
sample_sub['History of HeartDisease or Attack'] = y_pred['outcome']
sample_sub.to_csv("5hack_hdp_submission.csv", index=False)

print("Submission saved to 5hack_hdp_submission.csv")
sample_sub.head()

Submission saved to 5hack_hdp_submission.csv


,ID,History of HeartDisease or Attack
0,test_000001,No
1,test_000002,No
2,test_000003,Yes
3,test_000004,No
4,test_000005,No
